# Nevada superhot demonstration: 350 °C at 7 km

This notebook uses only public geoPFA interfaces and an explicit configuration cell. Local source data and generated outputs are deliberately not version-controlled. It propagates the Stanford Thermal Earth Model's mean and uncertainty into `P(T(7 km) > 350 °C)`. Seven kilometres is the shallowest released model depth that reaches 350 °C in the Nevada geoPFA footprint. No target-matched labels exist, so the heat component is explicitly prior-predictive and is compared with VoterVeto only as a contrast in map semantics.

In [ ]:
import copy
import os
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio

from geopfa.layer_combination import VoterVeto
from geopfa.prob import ProbabilisticConfig, run_probabilistic

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "pyproject.toml").exists():
            return current
        current = current.parent
    raise FileNotFoundError("geoPFA repository root not found")


def required_local_path(env_name: str, default: Path) -> Path:
    path = Path(os.environ.get(env_name, default)).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(
            f"Required local input is absent: {path}. Set {env_name} or prepare "
            "the documented public study data before running this notebook."
        )
    return path


def surface_on_raster_support(
    surface, *, raster_crs, raster_bounds, raster_resolution, name: str
):
    if surface.crs is None:
        raise ValueError(f"{name} surface must declare a CRS")
    if raster_crs is None or not raster_crs.is_projected:
        raise ValueError("thermal raster CRS must be projected")
    aligned = surface.to_crs(raster_crs)
    bounds = aligned.total_bounds
    if not np.isfinite(bounds).all():
        raise ValueError(f"{name} surface bounds must be finite")
    tolerance = max(abs(float(value)) for value in raster_resolution)
    outside = (
        bounds[0] < raster_bounds.left - tolerance
        or bounds[1] < raster_bounds.bottom - tolerance
        or bounds[2] > raster_bounds.right + tolerance
        or bounds[3] > raster_bounds.top + tolerance
    )
    if outside:
        raise ValueError(f"{name} surface falls outside the thermal raster support")
    return aligned

repo_root = find_repo_root(Path.cwd())
project_dir = repo_root / "examples" / "Nevada" / "2D"
output_root = Path(
    os.environ.get("GEOPFA_DEMO_OUTPUT_ROOT", project_dir / "outputs")
).expanduser().resolve()
output_dir = output_root / "nevada_superhot_350c_7km"
pfa_path = required_local_path(
    "GEOPFA_NEVADA_PFA", project_dir / "notebooks" / "fpa.pkl"
)
thermal_mean_path = required_local_path(
    "GEOPFA_NEVADA_7KM_MEAN",
    project_dir / "data" / "thermal" / "stanford_temperature_7km_mean_c.tif",
)
thermal_sd_path = required_local_path(
    "GEOPFA_NEVADA_7KM_SD",
    project_dir / "data" / "thermal" / "stanford_temperature_7km_sd_c.tif",
)
with pfa_path.open("rb") as stream:
    pfa = pickle.load(stream)

config_dict = {'enabled': True,
 'output_dir': '../outputs/superhot_350c_7km',
 'dimensions': '2d',
 'labels': {'observation_models': {'heat': {'family': 'gaussian'}}},
 'alpha': {'heat': {'mode': 'thermal_exceedance',
                    'thermal_raster': '../data/thermal/stanford_temperature_7km_mean_c.tif',
                    'uncertainty_raster': '../data/thermal/stanford_temperature_7km_sd_c.tif',
                    'threshold': 350.0,
                    'p_min': 1e-12,
                    'p_max': 0.999999999999,
                    'force_prior_predictive': True,
                    'use_evidence_prior': False}},
 'spatial_field': {'enabled': False},
 'inference': {'backend': 'gblk', 'gblk_bayesian': {'enabled': False}},
 'calibration': {'method': 'none'},
 'combination': {'rule': 'product'},
 'scenarios': [],
 'outputs': {'probability_rasters': True,
             'uncertainty_rasters': False,
             'posterior_draw_blocks': False,
             'format': ['geotiff', 'csv']}}
config_dict["output_dir"] = str(output_dir)
config_dict["alpha"]["heat"]["thermal_raster"] = str(thermal_mean_path)
config_dict["alpha"]["heat"]["uncertainty_raster"] = str(thermal_sd_path)
config = ProbabilisticConfig.from_dict(config_dict)
config

In [ ]:
pfa_vv = VoterVeto.do_voter_veto(
    copy.deepcopy(pfa),
    normalize_method="minmax",
    component_veto=False,
    criteria_veto=True,
    normalize=True,
    norm_to=5,
)
model_result = run_probabilistic(
    pfa,
    config,
    input_artifacts={"pfa_pickle": pfa_path},
)
probability = model_result.components["heat"].probability.copy()
with rasterio.open(thermal_mean_path) as source:
    temperature_mean = source.read(1, masked=True).astype(float).filled(np.nan)
    thermal_crs = source.crs
    thermal_bounds = source.bounds
    thermal_resolution = source.res
    map_extent = [source.bounds.left, source.bounds.right, source.bounds.bottom, source.bounds.top]
probability[["probability"]].describe()

In [ ]:
vv = pfa_vv["criteria"]["geologic"]["components"]["heat"]["pr_norm"]
vv_plot = surface_on_raster_support(
    vv,
    raster_crs=thermal_crs,
    raster_bounds=thermal_bounds,
    raster_resolution=thermal_resolution,
    name="VoterVeto",
)
probability_plot = surface_on_raster_support(
    probability,
    raster_crs=thermal_crs,
    raster_bounds=thermal_bounds,
    raster_resolution=thermal_resolution,
    name="probability",
)
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2), constrained_layout=True)
vv_plot.plot(column="favorability", cmap="viridis", vmin=0, vmax=5, markersize=2, legend=True, ax=axes[0])
temp_image = axes[1].imshow(temperature_mean, extent=map_extent, origin="upper", cmap="inferno")
probability_plot.plot(column="probability", cmap="magma", vmin=0, vmax=1, markersize=2, legend=True, ax=axes[2])
fig.colorbar(temp_image, ax=axes[1], label="Temperature (°C)")
axes[0].set_title("Traditional VoterVeto heat score")
axes[1].set_title("Stanford mean temperature at 7 km")
axes[2].set_title("P(T at 7 km > 350 °C)")
for axis in axes:
    axis.set_xlim(thermal_bounds.left, thermal_bounds.right)
    axis.set_ylim(thermal_bounds.bottom, thermal_bounds.top)
    axis.set_axis_off()
plt.show()

In [ ]:
{
    "target": "P(T at 7 km > 350 C)",
    "depth_selection": (
        "7 km is the shallowest released Stanford-model depth with nonzero "
        "350 C mean exceedance inside the demo footprint."
    ),
    "n_grid_cells": int(len(probability)),
    "temperature_mean_c": {
        "min": float(np.nanmin(temperature_mean)),
        "mean": float(np.nanmean(temperature_mean)),
        "max": float(np.nanmax(temperature_mean)),
    },
    "probability": probability.probability.describe().to_dict(),
    "calibration_status": (
        "Not estimable: the Great Basin temperature archive contains no 350 C "
        "observations. Uncertainty is inherited from the published thermal model."
    ),
}

## Interpretation and caveat

This is a regional deep-resource screening demonstration. It shows that geoPFA can consume a physical prior with uncertainty and return a probability map through the same config-driven workflow. It does not establish drillability, permeability, an economic resource, or empirical calibration at 350 °C.